# Train, Valid, Test Split (ml.m5.12xlarge)

In [1]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

### Functions

In [2]:
# denote df
def mark_df(flt_prop_row):
    if flt_prop_row <= 0.6:
        return 'train'
    elif flt_prop_row <= 0.8:
        return 'valid'
    else:
        return 'test'

### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')
str_dirname_output = './output'

str_id = 'uniqueid'
str_datecol = 'applicationdate__app'
str_target = 'target'

list_cols_id = [
    str_id,
    str_datecol,
    str_target,
]

Project: 20231010-gen-xii
Step: 03_train_valid_test_split


### Output

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

### Import data

In [5]:
%%time

# import
str_filename = 'df_raw.gzip'
str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/01_data_collection/output/{str_filename}'
# read from s3
df = pd.read_parquet(str_uri)
# replace
df.replace(['NaN','nan'], np.nan, inplace=True)

# show
df

<timed exec>:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


CPU times: user 15.9 s, sys: 33.5 s, total: 49.4 s
Wall time: 7.28 s


,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,dti__app,pti__app,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,RunningNetLoss,bitTarget24Months,target
0,20181103.0,20181103.0,NaN,201810.0,69614.0,2018-11-03,1.0,AU,2018-11-05,6847.0,...,0.327746,0.039116,0,0.910195,auto,0,2018-02-16 12:20:23.657,0.0,0,0
1,20181103.0,20181103.0,NaN,201810.0,69614.0,2018-11-03,1.0,AU,2018-11-05,6847.0,...,0.327746,0.039116,0,0.910195,auto,0,2018-02-16 12:20:23.657,0.0,0,0
2,20181103.0,20181103.0,NaN,201810.0,74719.0,2018-11-03,1.0,AU,2018-11-03,27323.0,...,0.486804,0.111285,0,1.049339,suv,1,2007-02-15 09:05:41.530,0.0,0,0
3,20181103.0,20181103.0,NaN,201810.0,74719.0,2018-11-03,1.0,AU,2018-11-03,27323.0,...,0.486804,0.111285,0,1.049339,suv,1,2007-02-15 09:05:41.530,0.0,0,0
4,20181030.0,20181030.0,NaN,201810.0,65604.0,2018-10-30,1.0,AU,2018-10-30,0.0,...,0.191788,0.071203,0,1.082675,truck,1,2011-05-20 14:53:37.283,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186170,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.466477,0.082986,0,0.976902,auto,1,2007-07-09 17:18:19.440,0.0,0,0
186171,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.466477,0.082986,0,0.976902,auto,1,2007-07-09 17:18:19.440,0.0,0,0
186172,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.352323,0.121456,0,1.149966,auto,0,2013-02-21 14:58:51.357,0.0,0,0
186173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.299975,0.062550,0,0.741723,suv,0,2011-09-20 11:51:00.783,0.0,0,0


### Mark each row as train, valid, or test

In [6]:
%%time

# sort
df.sort_values(by=str_datecol, ascending=True, inplace=True)

# create column
int_nrows = df.shape[0]
df['int_row'] = [i for i in range(1, int_nrows + 1)]
# divide by n rows
df['flt_prop_row'] = df['int_row'] / int_nrows

# drop
df.drop('int_row', axis=1, inplace=True)

# mark the df
df['data_set'] = df['flt_prop_row'].apply(mark_df)

# show
df

CPU times: user 1.52 s, sys: 1.8 s, total: 3.32 s
Wall time: 3.07 s


,dtmstampcreation__base,dtmapproved__base,dtmdeclined__base,observationdate__base,analyticsmatchkey__base,decision_dte__base,booked__base,acct_typ_cde__base,open_dte__base,curr_bal_amt__base,...,bitservicecontract__app,fltadvance__app,strvehicletype__app,bitgap__app,dealerstampcreation__app,RunningNetLoss,bitTarget24Months,target,flt_prop_row,data_set
38325,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1.106635,auto,0,2012-06-18 09:31:54.393,9070.52,1,1,0.000005,train
38324,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1.106635,auto,0,2012-06-18 09:31:54.393,9070.52,1,1,0.000011,train
38326,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1.254334,auto,1,2012-03-06 16:36:21.623,0.00,0,0,0.000016,train
38327,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1,1.254334,auto,1,2012-03-06 16:36:21.623,0.00,0,0,0.000021,train
38329,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0,1.249982,auto,0,2009-10-15 16:06:40.767,10250.98,1,1,0.000027,train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7864,20191231.0,20191231.0,NaN,201910.0,920826.0,2019-12-31,1.0,AU,2020-01-02,16196.0,...,1,1.210836,auto,1,2009-12-28 16:59:42.413,0.00,0,0,0.999979,test
7863,20191231.0,20191231.0,NaN,201910.0,920826.0,2019-12-31,1.0,AU,2020-01-02,16196.0,...,1,1.210836,auto,1,2009-12-28 16:59:42.413,0.00,0,0,0.999984,test
20360,20191231.0,20191231.0,NaN,201910.0,943817.0,2019-12-31,1.0,AU,2019-12-31,19634.0,...,1,0.862106,auto,1,2019-10-16 09:37:21.113,0.00,0,0,0.999989,test
20361,20191231.0,20191231.0,NaN,201910.0,943817.0,2019-12-31,1.0,AU,2019-12-31,19634.0,...,1,0.862106,auto,1,2019-10-16 09:37:21.113,0.00,0,0,0.999995,test


### Show value counts

In [7]:
ser_prop = pd.value_counts(df['data_set'], normalize=True)
print(ser_prop)

data_set
train    0.6
valid    0.2
test     0.2
Name: proportion, dtype: float64


/tmp/ipykernel_73506/1628931787.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  ser_prop = pd.value_counts(df['data_set'], normalize=True)


In [8]:
ser_freq = pd.value_counts(df['data_set'], normalize=False)
print(ser_freq)

data_set
train    111705
valid     37235
test      37235
Name: count, dtype: int64


/tmp/ipykernel_73506/3389291447.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  ser_freq = pd.value_counts(df['data_set'], normalize=False)


### Write data frames

In [9]:
list_df = [
    'train',
    'valid',
    'test',
]
for str_df in tqdm (list_df):
    # subset
    df_tmp = df[df['data_set']==str_df].copy()
    
    # drop
    list_cols = [
        'flt_prop_row',
        'data_set',
    ]
    df_tmp.drop(list_cols, axis=1, inplace=True)
    
    # write to s3
    str_filename = f'df_{str_df}_raw.gzip'
    str_uri = f's3://{str_project}/02_pricing_pd/01_data_prep/{str_step}/{str_filename}'
    df_tmp.to_parquet(str_uri, compression='gzip')

100%|██████████| 3/3 [01:12<00:00, 24.05s/it]
